# PlagIA Enterprise · 01 — Gobernanza de Datos, Auditoría de Leakage y Manifiesto Multinivel

**Subsistema de Ingeniería de Datos y MLOps (`ml_engine`)**

### Objetivos Técnicos
1. Construir el **manifiesto unificado** para capturas de drones (VANTs) y dispositivos móviles en terreno desde `dataset/`.
2. Ejecutar la **auditoría de integridad** para cuantificar archivos duplicados (`- Copy.jpg`) y fuentes de aumentación cruzadas (`augmented__<id>_`) entre `Train/` y `Test/`.
3. Ejecutar la ingesta distribuida con **PySpark (`build_manifest_spark`)** preparada para Data Lakes.
4. Generar el subconjunto balanceado y el particionado **70% Train / 15% Val / 15% Test** agrupado por `group_id` con georreferenciación en los distritos arroceros del Tolima.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "pyproject.toml").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import matplotlib.pyplot as plt

from ml_engine.config import (
    DRONE_DATASET_DIR,
    FULL_MANIFEST_PATH,
    SEED,
    SPLIT_MANIFEST_PATH,
    SUBSET_PER_CLASS,
)
from ml_engine.data_governance import (
    audit_data_leakage,
    build_drone_manifest,
    build_multilevel_manifest,
    make_balanced_subset,
    split_manifest_grouped,
)

print("Raíz del monorepo PlagIA:", PROJECT_ROOT)
print("Dataset Dron VANT       :", DRONE_DATASET_DIR)

## 1. Auditoría de Calidad de Datos y Detección de Data Leakage

In [ ]:
audit_report = audit_data_leakage(DRONE_DATASET_DIR)
for k, v in audit_report.items():
    print(f"{k:32s}: {v}")

## 2. Construcción del Manifiesto Limpio (Dron VANT y Multinivel)

In [ ]:
drone_manifest = build_drone_manifest(DRONE_DATASET_DIR, exclude_copies=True)
multilevel_manifest = build_multilevel_manifest(exclude_copies=True)

print(f"Total registros Dron VANT (sin copias): {len(drone_manifest)}")
print(f"Total registros Multinivel (Dron + Móvil): {len(multilevel_manifest)}")
print("\nDistribución por clase (Dron VANT):")
print(drone_manifest["label_name"].value_counts())
drone_manifest.head()

## 3. Ingesta Distribuida con PySpark (Escalabilidad sobre Data Lake)

In [ ]:
try:
    from ml_engine.data_governance import build_manifest_spark
    sdf = build_manifest_spark(DRONE_DATASET_DIR)
    print("Total registros leídos con PySpark:", sdf.count())
    sdf.groupBy("label_name").count().show()
except Exception as exc:
    print(f"Aviso PySpark local ({type(exc).__name__}): se continúa con el manifiesto Pandas validado.")

## 4. Muestreo Balanceado y Split 70/15/15 Agrupado sin Fuga de Datos

In [ ]:
subset_df = make_balanced_subset(drone_manifest, per_class=SUBSET_PER_CLASS, seed=SEED)
split_df = split_manifest_grouped(subset_df, seed=SEED)

leak_check = split_df.groupby("group_id")["split"].nunique()
n_leaked = int((leak_check > 1).sum())
print("Grupos compartidos entre múltiples splits (debe ser 0):", n_leaked)
assert n_leaked == 0, "Fuga de datos detectada en el split"

print("\nComposición del Split 70/15/15:")
print(split_df.groupby(["split", "label_name"]).size())

FULL_MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
multilevel_manifest.to_csv(FULL_MANIFEST_PATH, index=False)
split_df.to_csv(SPLIT_MANIFEST_PATH, index=False)
print("\nManifiestos guardados en:", SPLIT_MANIFEST_PATH)